# NB15 — Fourth dataset: DeepFake Detection Dataset (DFD) coupling + monitoring

External-validity test of the competence-calibration coupling on an independently-sourced
dataset (Google/Jigsaw DFD, distributed via the FaceForensics++ download script). DFD has real
actors and DeepFakes manipulations at three compression levels (c0 raw, c23, c40), which gives a
spread of detector competence to correlate calibration against.

Pipeline (same as the FF++/DF40 work):
1. Download DFD subset via FaceForensics download script (--server EU2).
2. Crop faces (dlib HOG) -> frames, same preprocessing as the main experiments.
3. Score with Xception-FS. Build per-condition cells (compression x manipulation) for an AUC spread.
4. Per-condition: AUC, ECE_cal (oracle per-condition calibration), reference-free signals.
5. Coupling: AUC vs ECE_cal across conditions. Compare to FF++/DF40 (r ~ -0.8).

Checkpointed. Saves to reports/calibration/coupling_dfd_xceptionFS.csv.


## Cell 1 — Download DFD via FaceForensics script (EU2 server)

The FF++ download script (downloaded from the link in the access email) pulls DFD with the
'DeepFakeDetection' dataset name. We fetch the manipulated (DeepFakes) + original actor videos
at c23 and c40 compression. Adjust NUM_VIDEOS for subset size.


In [3]:
import subprocess, os
CDTS = "/content/drive/MyDrive/CDTS_Research"
os.makedirs(CDTS, exist_ok=True)
SCRIPT_PATH = f"{CDTS}/faceforensics_download_v4.py"

# fetch the FF++ v4 download script from the official TUM server
r = subprocess.run(
    f'wget -q "https://kaldir.vc.in.tum.de/faceforensics_download_v4.py" -O "{SCRIPT_PATH}"',
    shell=True, capture_output=True, text=True
)
print(r.stderr if r.stderr else "downloaded")

# verify it landed and is a real python script
if os.path.exists(SCRIPT_PATH) and os.path.getsize(SCRIPT_PATH) > 1000:
    print(f"✓ script saved: {SCRIPT_PATH} ({os.path.getsize(SCRIPT_PATH)} bytes)")
    print("\n=== first 30 lines (confirm it's the real script) ===")
    subprocess.run(f"head -30 {SCRIPT_PATH}", shell=True)
    print("\n=== usage/help ===")
    subprocess.run(f"python {SCRIPT_PATH} --help", shell=True)
else:
    print("✗ download failed or file too small — check the URL / network")

downloaded
✓ script saved: /content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py (10727 bytes)

=== first 30 lines (confirm it's the real script) ===

=== usage/help ===


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, subprocess, sys
CDTS = "/content/drive/MyDrive/CDTS_Research"
REPO = f"{CDTS}/deepfake-trust-research"
DFD_ROOT = f"{CDTS}/DFD"          # download target on Drive (survives resets)
os.makedirs(DFD_ROOT, exist_ok=True)

# ---- the FF++ download script. If you've saved it to Drive, point to it; else we fetch usage. ----
# Common locations to check:
SCRIPT=None
import glob
for cand in glob.glob(f"{CDTS}/**/faceforensics_download*.py", recursive=True) + \
            glob.glob(f"{CDTS}/**/download*.py", recursive=True) + \
            glob.glob(f"/content/**/faceforensics_download*.py", recursive=True):
    SCRIPT=cand; break
print("download script:", SCRIPT if SCRIPT else "NOT FOUND - upload it to Drive or /content")
if SCRIPT:
    # show its help so we use exact args
    print("\n=== script --help ===")
    subprocess.run(f"python {SCRIPT} --help", shell=True)
else:
    print("\n>>> Upload faceforensics_download_v4.py to Drive (e.g. CDTS_Research/) and re-run.")
    print(">>> It came from the link in your FaceForensics access email.")

Mounted at /content/drive
download script: /content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py

=== script --help ===


## Cell 2 — Run the DFD download (DeepFakeDetection dataset, c23 + c40)

DFD dataset name in the script is 'DeepFakeDetection' (manipulated) and
'DeepFakeDetection_original' (real actor videos). We pull both at two compressions for a
competence spread. NUM controls subset size for a fast coupling test.


In [5]:
import subprocess, os, glob
CDTS = "/content/drive/MyDrive/CDTS_Research"
DFD_ROOT = f"{CDTS}/DFD"
# locate script again
SCRIPT=None
for cand in glob.glob(f"{CDTS}/**/faceforensics_download*.py", recursive=True) + glob.glob(f"{CDTS}/**/download*.py", recursive=True):
    SCRIPT=cand; break
assert SCRIPT, "download script not found - upload it first (Cell 1)"

# DFD = 'DeepFakeDetection' (fakes) + 'DeepFakeDetection_original' (real). EU2 server mandatory.
# -d dataset, -c compression, --server EU2. The script downloads into DFD_ROOT.
# We do c23 (standard) and c40 (heavy compression) to span competence.
for dataset in ["DeepFakeDetection_original", "DeepFakeDetection"]:
    for comp in ["c23", "c40"]:
        print(f"\n=== downloading {dataset} {comp} ===")
        cmd = f"python {SCRIPT} {DFD_ROOT} -d {dataset} -c {comp} --server EU2"
        print(cmd)
        # NOTE: first run will prompt EULA acceptance; if it hangs on input, add 'yes |' prefix
        subprocess.run(f"yes | {cmd}", shell=True)

print("\n=== downloaded structure ===")
subprocess.run(f"find {DFD_ROOT} -name '*.mp4' | head -20", shell=True)
subprocess.run(f"echo 'total videos:'; find {DFD_ROOT} -name '*.mp4' | wc -l", shell=True)


=== downloading DeepFakeDetection_original c23 ===
python /content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py /content/drive/MyDrive/CDTS_Research/DFD -d DeepFakeDetection_original -c c23 --server EU2

=== downloading DeepFakeDetection_original c40 ===
python /content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py /content/drive/MyDrive/CDTS_Research/DFD -d DeepFakeDetection_original -c c40 --server EU2


KeyboardInterrupt: 

In [6]:
import subprocess
DFD = "/content/drive/MyDrive/CDTS_Research/DFD"
subprocess.run(f"find {DFD} -name '*.mp4' | head -20", shell=True)
print("---")
subprocess.run(f"find {DFD} -name '*.mp4' | wc -l", shell=True)
subprocess.run(f"du -sh {DFD} 2>/dev/null", shell=True)
print("--- breakdown by folder ---")
subprocess.run(f"find {DFD} -name '*.mp4' | sed 's|/[^/]*$||' | sort | uniq -c", shell=True)

---
--- breakdown by folder ---


CompletedProcess(args="find /content/drive/MyDrive/CDTS_Research/DFD -name '*.mp4' | sed 's|/[^/]*$||' | sort | uniq -c", returncode=0)

In [7]:
import subprocess, os
CDTS = "/content/drive/MyDrive/CDTS_Research"

# 1. Does the DFD folder even exist? What's in it (any files at all, not just mp4)?
print("=== DFD folder contents (everything, not just mp4) ===")
subprocess.run(f"ls -la {CDTS}/DFD 2>&1", shell=True)
subprocess.run(f"find {CDTS}/DFD -type f 2>/dev/null | head -30", shell=True)
print("=== any files anywhere under DFD ===")
subprocess.run(f"find {CDTS}/DFD -type f 2>/dev/null | wc -l", shell=True)

# 2. Did anything download to LOCAL disk instead (wrong path)?
print("=== check /content for stray downloads ===")
subprocess.run("find /content -name '*.mp4' 2>/dev/null | grep -v drive | head -10", shell=True)

# 3. Test: can we even reach the TUM server? (this is the real question)
print("=== can we reach the download server? ===")
subprocess.run("timeout 20 curl -sI https://kaldir.vc.in.tum.de/faceforensics_download_v4.py 2>&1 | head -5", shell=True)

=== DFD folder contents (everything, not just mp4) ===
=== any files anywhere under DFD ===
=== check /content for stray downloads ===
=== can we reach the download server? ===


CompletedProcess(args='timeout 20 curl -sI https://kaldir.vc.in.tum.de/faceforensics_download_v4.py 2>&1 | head -5', returncode=0)

In [8]:
import subprocess
print("=== google (should work if internet is up) ===")
subprocess.run("timeout 15 curl -sI https://www.google.com 2>&1 | head -3", shell=True)
print("=== github (should work) ===")
subprocess.run("timeout 15 curl -sI https://github.com 2>&1 | head -3", shell=True)
print("=== TUM server again ===")
subprocess.run("timeout 25 curl -v https://kaldir.vc.in.tum.de/ 2>&1 | tail -15", shell=True)

=== google (should work if internet is up) ===
=== github (should work) ===
=== TUM server again ===


CompletedProcess(args='timeout 25 curl -v https://kaldir.vc.in.tum.de/ 2>&1 | tail -15', returncode=0)

In [9]:
import subprocess
r = subprocess.run("timeout 15 curl -sI https://www.google.com", shell=True, capture_output=True, text=True)
print("RETURN CODE:", r.returncode)
print("STDOUT:", repr(r.stdout))
print("STDERR:", repr(r.stderr))

RETURN CODE: 0
STDOUT: 'HTTP/2 200 \ncontent-type: text/html; charset=ISO-8859-1\ncontent-security-policy-report-only: object-src \'none\';base-uri \'self\';script-src \'nonce-2ytauYHHt3px0v1MZOBARw\' \'strict-dynamic\' \'report-sample\' \'unsafe-eval\' \'unsafe-inline\' https: http:;report-uri https://csp.withgoogle.com/csp/gws/other-hp\naccept-ch: Sec-CH-Prefers-Color-Scheme\np3p: CP="This is not a P3P policy! See g.co/p3phelp for more info."\ndate: Sun, 28 Jun 2026 09:40:47 GMT\nserver: gws\nx-xss-protection: 0\nx-frame-options: SAMEORIGIN\nexpires: Sun, 28 Jun 2026 09:40:47 GMT\ncache-control: private\nset-cookie: AEC=AdJVEaub8BXiKwG7XVRLA2H7NSYIm_5Ya73XNupp_ZgMR1_ClnWar0vj3A; expires=Fri, 25-Dec-2026 09:40:47 GMT; path=/; domain=.google.com; Secure; HttpOnly; SameSite=lax\nset-cookie: NID=532=aydXZ3SGXTXIwbVZ1EuvENK057jNMld1o3es2Lo6rbnxBOYGciOQGBoVe69gFNMM5lzcaH_SMJYlB4J4zof-S5dsVdPrv4Mo-_aAduajgL61sBN5r6eXKEc9q_j196zCsUWVyWXIp7RtaVDVNZFfQ8dREZhZGvP1Xb9QP51h0p1uqHAsHMBYG-OIewutsqW

In [10]:
import subprocess
r = subprocess.run("timeout 30 curl -sI https://kaldir.vc.in.tum.de/faceforensics_download_v4.py",
                   shell=True, capture_output=True, text=True)
print("RETURN CODE:", r.returncode)
print("STDOUT:", repr(r.stdout))
print("STDERR:", repr(r.stderr))

RETURN CODE: 0
STDOUT: 'HTTP/1.1 200 OK\nDate: Sun, 28 Jun 2026 09:43:11 GMT\nServer: Apache/2.4.52 (Ubuntu)\nLast-Modified: Wed, 15 Jul 2020 12:57:31 GMT\nETag: "29e7-5aa7a778986ed"\nAccept-Ranges: bytes\nContent-Length: 10727\nAccess-Control-Allow-Origin: *\nContent-Type: text/x-python\n\n'
STDERR: ''


In [11]:
import subprocess, os, time

SCRIPT = "/content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py"
LOCAL  = "/content/DFD_local"          # LOCAL disk, NOT Drive
os.makedirs(LOCAL, exist_ok=True)

# Probe: just 3 DFD fake videos, c23, local disk. Watch the clock.
print("=== PROBE: 3 DeepFakeDetection videos to LOCAL disk ===")
t0 = time.time()
r = subprocess.run(
    f"python {SCRIPT} {LOCAL} -d DeepFakeDetection -c c23 -n 3 --server EU2",
    shell=True, capture_output=True, text=True, timeout=600   # hard 10-min ceiling
)
print(f"elapsed: {time.time()-t0:.0f}s   return: {r.returncode}")
print("STDOUT:", r.stdout[-2000:])
print("STDERR:", r.stderr[-1000:])
print("=== files landed ===")
subprocess.run(f"find {LOCAL} -name '*.mp4' -exec ls -lh {{}} +", shell=True)

=== PROBE: 3 DeepFakeDetection videos to LOCAL disk ===
elapsed: 0s   return: 1
STDOUT: By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://kaldir.vc.in.tum.de/faceforensics/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.

STDERR: Traceback (most recent call last):
  File "/content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py", line 261, in <module>
    main(args)
  File "/content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py", line 144, in main
    _ = input('')
        ^^^^^^^^^
EOFError: EOF when reading a line

=== files landed ===


CompletedProcess(args="find /content/DFD_local -name '*.mp4' -exec ls -lh {} +", returncode=0)

In [12]:
import subprocess, os, time

SCRIPT = "/content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py"
LOCAL  = "/content/DFD_local"
os.makedirs(LOCAL, exist_ok=True)

print("=== PROBE: 3 DeepFakeDetection videos to LOCAL disk ===")
t0 = time.time()
r = subprocess.run(
    f"echo '' | python {SCRIPT} {LOCAL} -d DeepFakeDetection -c c23 -n 3 --server EU2",
    shell=True, capture_output=True, text=True, timeout=600
)
print(f"elapsed: {time.time()-t0:.0f}s   return: {r.returncode}")
print("STDOUT:", r.stdout[-2500:])
print("STDERR:", r.stderr[-1000:])
print("=== files landed ===")
subprocess.run(f"find {LOCAL} -name '*.mp4' -exec ls -lh {{}} +", shell=True)

=== PROBE: 3 DeepFakeDetection videos to LOCAL disk ===
elapsed: 61s   return: 0
STDOUT: By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://kaldir.vc.in.tum.de/faceforensics/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.
Output path: /content/DFD_local/manipulated_sequences/DeepFakeDetection/c23/videos

STDERR: 
100%|██████████| 3/3 [00:54<00:00, 18.08s/it]

=== files landed ===


CompletedProcess(args="find /content/DFD_local -name '*.mp4' -exec ls -lh {} +", returncode=0)

In [13]:
import subprocess, os, time

SCRIPT = "/content/drive/MyDrive/CDTS_Research/faceforensics_download_v4.py"
LOCAL  = "/content/DFD_local"
N = 40

for dataset, label in [("DeepFakeDetection", "FAKES"), ("DeepFakeDetection_original", "REALS")]:
    print(f"\n=== {label}: {dataset}, c23, first {N} videos -> LOCAL disk ===")
    t0 = time.time()
    r = subprocess.run(
        f"echo '' | python {SCRIPT} {LOCAL} -d {dataset} -c c23 -n {N} --server EU2",
        shell=True, capture_output=True, text=True, timeout=3600
    )
    print(f"  elapsed: {time.time()-t0:.0f}s   return: {r.returncode}")
    if r.returncode != 0:
        print("  STDERR:", r.stderr[-800:])
    # tqdm writes to stderr; show the final line
    tail = [l for l in r.stderr.strip().split("\n") if l.strip()]
    if tail: print("  last progress:", tail[-1])

# inventory — with capture so we actually SEE it
print("\n=== FINAL INVENTORY ===")
inv = subprocess.run(f"find {LOCAL} -name '*.mp4'", shell=True, capture_output=True, text=True)
files = [f for f in inv.stdout.strip().split("\n") if f]
print(f"total .mp4 files: {len(files)}")
by_dir = {}
for f in files:
    d = "/".join(f.split("/")[:-1]).replace(LOCAL, "")
    by_dir[d] = by_dir.get(d, 0) + 1
for d, c in sorted(by_dir.items()):
    print(f"  {c:3d}  {d}")
du = subprocess.run(f"du -sh {LOCAL}", shell=True, capture_output=True, text=True)
print("disk used:", du.stdout.strip())


=== FAKES: DeepFakeDetection, c23, first 40 videos -> LOCAL disk ===
  elapsed: 1064s   return: 0
  last progress: 100%|██████████| 40/40 [17:39<00:00, 26.48s/it]

=== REALS: DeepFakeDetection_original, c23, first 40 videos -> LOCAL disk ===
  elapsed: 740s   return: 0
  last progress: 100%|██████████| 40/40 [12:18<00:00, 18.46s/it]

=== FINAL INVENTORY ===
total .mp4 files: 80
   40  /manipulated_sequences/DeepFakeDetection/c23/videos
   40  /original_sequences/actors/c23/videos
disk used: 489M	/content/DFD_local


## Cell 3 — Crop faces (dlib HOG), same preprocessing as FF++/DF40

Extract frames from each video, detect+crop the face with dlib HOG, resize, save as PNG.
Organized by condition (dataset x compression) so we can compute per-condition AUC/ECE.


In [14]:
# ============================================================================
# NB15 (continued) — DFD frame extraction + dlib HOG face cropping
# Matches the FF++/DF40 pipeline: dlib HOG detector, square crop around the
# face box with margin, saved as .../frames/<video_id>/<NNN>.png
#
# RUN CELL A FIRST (3-video probe). Only run CELL B once the probe looks right.
# ============================================================================

# ----------------------------------------------------------------------------
# CELL A — PROBE: crop 3 videos, eyeball the output before committing to all 80
# ----------------------------------------------------------------------------
import os, glob, cv2, dlib, numpy as np
from pathlib import Path

# --- config (match your FF++/DF40 settings) ---
CROP_SIZE   = 256          # 256x256 to match the CNN scoring resolution; CLIP resizes to 224 at score time
MARGIN      = 0.30         # fraction of box size added around the dlib face box (DeepfakeBench-style)
FRAMES_PER_VIDEO = 32      # sample this many evenly-spaced frames per video (DF40 used a similar density)
LOCAL = "/content/DFD_local"
CROP_ROOT = "/content/DFD_crops"      # crops go LOCAL first; copy only these to Drive at the end

FAKE_DIR = f"{LOCAL}/manipulated_sequences/DeepFakeDetection/c23/videos"
REAL_DIR = f"{LOCAL}/original_sequences/actors/c23/videos"

detector = dlib.get_frontal_face_detector()   # HOG detector, same as FF++ pipeline

def crop_face_from_frame(frame_bgr):
    """Detect the largest face with dlib HOG, return a square margin-padded CROP_SIZE crop, or None."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    dets = detector(rgb, 1)
    if len(dets) == 0:
        return None
    # largest face
    d = max(dets, key=lambda r: (r.right()-r.left())*(r.bottom()-r.top()))
    x1, y1, x2, y2 = d.left(), d.top(), d.right(), d.bottom()
    w, h = x2-x1, y2-y1
    # add margin, make square
    cx, cy = x1 + w/2, y1 + h/2
    size = int(max(w, h) * (1 + MARGIN))
    nx1, ny1 = int(cx - size/2), int(cy - size/2)
    nx2, ny2 = nx1 + size, ny1 + size
    H, W = frame_bgr.shape[:2]
    # clamp to image, pad if needed
    pad_l = max(0, -nx1); pad_t = max(0, -ny1)
    pad_r = max(0, nx2-W); pad_b = max(0, ny2-H)
    nx1, ny1 = max(0, nx1), max(0, ny1)
    nx2, ny2 = min(W, nx2), min(H, ny2)
    crop = frame_bgr[ny1:ny2, nx1:nx2]
    if pad_l or pad_t or pad_r or pad_b:
        crop = cv2.copyMakeBorder(crop, pad_t, pad_b, pad_l, pad_r, cv2.BORDER_CONSTANT, value=[0,0,0])
    if crop.size == 0:
        return None
    return cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA)

def process_video(video_path, out_dir, n_frames=FRAMES_PER_VIDEO):
    """Extract n evenly-spaced frames, crop the face from each, save as <NNN>.png. Returns #saved."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return 0
    idxs = np.linspace(0, total-1, min(n_frames, total)).astype(int)
    saved = 0
    for j, fi in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
        ok, frame = cap.read()
        if not ok: continue
        crop = crop_face_from_frame(frame)
        if crop is None: continue
        cv2.imwrite(f"{out_dir}/{j:03d}.png", crop)
        saved += 1
    cap.release()
    return saved

# --- PROBE: 3 fake videos ---
fake_videos = sorted(glob.glob(f"{FAKE_DIR}/*.mp4"))[:3]
print(f"PROBE on {len(fake_videos)} fake videos (crop size {CROP_SIZE}, {FRAMES_PER_VIDEO} frames/video)")
import time
t0 = time.time()
for v in fake_videos:
    vid = Path(v).stem                                  # e.g. 01_02__exit_phone_room__YVGY8LOK
    out = f"{CROP_ROOT}/fake/frames/{vid}"
    n = process_video(v, out)
    print(f"  {vid[:40]:<42} -> {n} crops")
print(f"probe elapsed: {time.time()-t0:.0f}s for 3 videos")
print(f"\nInspect a crop visually before running CELL B:")
print(f"  from PIL import Image; Image.open(sorted(glob.glob('{CROP_ROOT}/fake/frames/*/000.png'))[0])")
print(f"\nIf crops look like properly centered faces at {CROP_SIZE}x{CROP_SIZE}, run CELL B.")
print(f"If faces are off-center / too tight / too loose, adjust MARGIN and re-probe.")


# ----------------------------------------------------------------------------
# CELL B — FULL CROP: all 80 videos (40 fake + 40 real) to LOCAL disk
#   Run ONLY after the Cell A probe looks right.
# ----------------------------------------------------------------------------
import time

def crop_all(video_dir, label_subdir, n_frames=FRAMES_PER_VIDEO):
    vids = sorted(glob.glob(f"{video_dir}/*.mp4"))
    print(f"\n=== cropping {len(vids)} videos -> {CROP_ROOT}/{label_subdir} ===")
    t0 = time.time()
    total_crops = 0; no_face = []
    for i, v in enumerate(vids):
        vid = Path(v).stem
        out = f"{CROP_ROOT}/{label_subdir}/frames/{vid}"
        n = process_video(v, out, n_frames)
        total_crops += n
        if n == 0: no_face.append(vid)
        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(vids)} videos, {total_crops} crops, {time.time()-t0:.0f}s")
    print(f"  DONE: {len(vids)} videos -> {total_crops} crops in {time.time()-t0:.0f}s")
    if no_face:
        print(f"  WARNING: {len(no_face)} videos yielded 0 crops (no face detected): {no_face[:5]}")
    return total_crops

n_fake = crop_all(FAKE_DIR, "fake")
n_real = crop_all(REAL_DIR, "real")

# --- inventory ---
print("\n=== CROP INVENTORY ===")
fake_crops = glob.glob(f"{CROP_ROOT}/fake/frames/*/*.png")
real_crops = glob.glob(f"{CROP_ROOT}/real/frames/*/*.png")
print(f"fake crops: {len(fake_crops)}  across {len(set(Path(p).parent for p in fake_crops))} videos")
print(f"real crops: {len(real_crops)}  across {len(set(Path(p).parent for p in real_crops))} videos")
import subprocess
du = subprocess.run(f"du -sh {CROP_ROOT}", shell=True, capture_output=True, text=True)
print(f"crop disk: {du.stdout.strip()}")
print(f"\nCrops ready. Next: the SCORING cell (needs your DeepfakeBench model loaders).")
print(f"Crops stay on LOCAL disk; we copy only the score parquets (tiny) to Drive, not the crops.")

PROBE on 3 fake videos (crop size 256, 32 frames/video)
  01_11__talking_against_wall__9229VVZ3      -> 32 crops
  01_27__outside_talking_still_laughing__Z   -> 32 crops
  02_15__talking_against_wall__HTG660F8      -> 32 crops
probe elapsed: 124s for 3 videos

Inspect a crop visually before running CELL B:
  from PIL import Image; Image.open(sorted(glob.glob('/content/DFD_crops/fake/frames/*/000.png'))[0])

If crops look like properly centered faces at 256x256, run CELL B.
If faces are off-center / too tight / too loose, adjust MARGIN and re-probe.

=== cropping 40 videos -> /content/DFD_crops/fake ===
  10/40 videos, 318 crops, 409s
  20/40 videos, 628 crops, 826s
  30/40 videos, 946 crops, 1241s
  40/40 videos, 1258 crops, 1651s
  DONE: 40 videos -> 1258 crops in 1651s

=== cropping 40 videos -> /content/DFD_crops/real ===
  10/40 videos, 306 crops, 418s
  20/40 videos, 615 crops, 833s
  30/40 videos, 921 crops, 1256s
  40/40 videos, 1235 crops, 1667s
  DONE: 40 videos -> 1235 crops 

## Cell 4 — Load Xception-FS, score DFD conditions

In [ ]:
import os, sys, glob, subprocess, importlib.util
CDTS="/content/drive/MyDrive/CDTS_Research"; REPO=f"{CDTS}/deepfake-trust-research"; DFB=f"{REPO}/external/DeepfakeBench"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"{CDTS}/{f}"): subprocess.run(f'cp "{CDTS}/{f}" /root/{f}', shell=True)
subprocess.run("pip -q install efficientnet_pytorch timm einops kornia simplejson", shell=True)
for k in list(sys.modules.keys()):
    if k.startswith("detectors") or k.startswith("networks") or k=="metrics" or k.startswith("metrics.") or k=="inference": del sys.modules[k]
for p in (f"{DFB}/training",DFB,f"{REPO}/src"):
    if p in sys.path: sys.path.remove(p)
sys.path.insert(0,DFB);sys.path.insert(0,f"{DFB}/training");sys.path.append(f"{REPO}/src")
spec=importlib.util.spec_from_file_location("inference",f"{REPO}/src/inference.py")
inference=importlib.util.module_from_spec(spec);sys.modules["inference"]=inference;spec.loader.exec_module(inference)
model,device,info=inference.load_detector(dfb_root=DFB,backbone_name="xception",ckpt_path=f"{REPO}/weights/train_on_fs/xception.pth")
model.eval();print("Xception-FS loaded:",info)

In [16]:
!pip -q install loralib

In [17]:
# ============================================================================
# NB15 (continued) — Score DFD crops with all 3 detectors + compute the coupling
#
# Reuses YOUR inference.py (load_detector + score_manifest) so DFD scores are
# on the IDENTICAL scale as the DF40 parquets. No new scoring logic.
#
# Run order: CELL C (build manifest) -> CELL D (score 3 detectors) ->
#            CELL E (within-DFD coupling).
# ============================================================================

# ----------------------------------------------------------------------------
# CELL C — build the DFD manifest DataFrame that score_manifest() expects
# ----------------------------------------------------------------------------
import os, sys, glob, importlib.util, subprocess
import pandas as pd, numpy as np
from pathlib import Path

CDTS = "/content/drive/MyDrive/CDTS_Research"
REPO = f"{CDTS}/deepfake-trust-research"
DFB  = f"{REPO}/external/DeepfakeBench"
CROP_ROOT = "/content/DFD_crops"          # from the crop cell
OUT_SCORES = f"{REPO}/reports/scores"     # DFD parquets go here (same dir as DF40)
os.makedirs(OUT_SCORES, exist_ok=True)

# git creds + deps (same bootstrap as your loader cell)
for f in [".gitconfig", ".git-credentials"]:
    if os.path.exists(f"{CDTS}/{f}"): subprocess.run(f'cp "{CDTS}/{f}" /root/{f}', shell=True)
subprocess.run("pip -q install efficientnet_pytorch timm einops kornia simplejson", shell=True)

# Build manifest. DFD crops live at:
#   {CROP_ROOT}/fake/frames/<video_id>/<NNN>.png   (label 1)
#   {CROP_ROOT}/real/frames/<video_id>/<NNN>.png   (label 0)
# DFD fake video_ids look like '01_11__talking_against_wall__9229VVZ3' where the
# leading '01' / '01_11' encodes actor identity -> we use it for identity slicing.
rows = []
for lab, sub in [(1, "fake"), (0, "real")]:
    for png in glob.glob(f"{CROP_ROOT}/{sub}/frames/*/*.png"):
        vid = Path(png).parent.name
        # actor id = leading token before first '_' for reals ('01'), or first two for fakes
        # DFD fakes: 'SRC_TGT__scene__hash' -> identity pair; use SRC actor as identity
        actor = vid.split("_")[0]
        rows.append({
            "frame_path": png,
            "label": lab,
            "video_id": vid,
            "identity_id": actor,                 # actor token, for per-identity slicing
            "method": "DFD",                      # single dataset tag; we slice differently below
            "split": "test",
        })
manifest = pd.DataFrame(rows)
print(f"DFD manifest: {len(manifest)} crops  ({(manifest.label==1).sum()} fake, {(manifest.label==0).sum()} real)")
print(f"unique fake videos: {manifest[manifest.label==1]['video_id'].nunique()}, "
      f"unique real videos: {manifest[manifest.label==0]['video_id'].nunique()}")
print(f"unique actors: {manifest['identity_id'].nunique()}")
manifest.to_parquet(f"{OUT_SCORES}/dfd_manifest.parquet")


# ----------------------------------------------------------------------------
# CELL D — score DFD with all 3 detectors (reusing inference.py)
# ----------------------------------------------------------------------------
# fresh import of inference.py (same pattern as your loader cell)
for k in list(sys.modules.keys()):
    if k.startswith("detectors") or k.startswith("networks") or k == "metrics" or k.startswith("metrics.") or k == "inference":
        del sys.modules[k]
for p in (f"{DFB}/training", DFB, f"{REPO}/src"):
    if p in sys.path: sys.path.remove(p)
sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")
spec = importlib.util.spec_from_file_location("inference", f"{REPO}/src/inference.py")
inference = importlib.util.module_from_spec(spec); sys.modules["inference"] = inference
spec.loader.exec_module(inference)

# Detector configs — CONFIRM these paths/resolutions match what scored DF40.
# (Xception path confirmed from your loader cell; EffNet/CLIP assumed same convention.)
DETECTORS = [
    # (tag, backbone_name, ckpt_path, resolution)
    ("xceptionFS",  "xception",       f"{REPO}/weights/train_on_fs/xception.pth",       256),
    ("effnetb4FS",  "efficientnetb4", f"{REPO}/weights/train_on_fs/efficientnetb4.pth", 256),
    ("clipFS",      "clip",           f"{REPO}/weights/train_on_fs/clip.pth",           224),  # CLIP at 224
]

for tag, backbone, ckpt, res in DETECTORS:
    print(f"\n=== scoring DFD with {tag} ({backbone}, res {res}) ===")
    if not os.path.exists(ckpt):
        print(f"  !! checkpoint not found: {ckpt}  -- skipping (fix path)")
        continue
    model, device, info = inference.load_detector(dfb_root=DFB, backbone_name=backbone, ckpt_path=ckpt)
    model.eval(); print(f"  loaded: {info}")
    scores = inference.score_manifest(model, device, manifest, batch_size=64, res=res, verbose=True)
    out_path = f"{OUT_SCORES}/{tag}_DFD.parquet"
    scores.to_parquet(out_path)
    aucs = inference.quick_auc(scores)
    print(f"  saved {out_path}  ({len(scores)} frames)  overall AUC={aucs.get('overall', float('nan')):.3f}")
    del model
    import torch, gc; gc.collect(); torch.cuda.empty_cache()

print("\nAll 3 detectors scored. Next: CELL E (within-DFD coupling).")


DFD manifest: 2493 crops  (1258 fake, 1235 real)
unique fake videos: 40, unique real videos: 40
unique actors: 27

=== scoring DFD with xceptionFS (xception, res 256) ===
  loaded: {'missing': 0, 'unexpected': 0}
  scored 64/2493
  scored 704/2493
  scored 1344/2493
  scored 1984/2493
  saved /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/reports/scores/xceptionFS_DFD.parquet  (2493 frames)  overall AUC=0.613

=== scoring DFD with effnetb4FS (efficientnetb4, res 256) ===
  loaded: {'missing': 0, 'unexpected': 0}
  scored 64/2493
  scored 704/2493
  scored 1344/2493
  scored 1984/2493
  saved /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/reports/scores/effnetb4FS_DFD.parquet  (2493 frames)  overall AUC=0.630

=== scoring DFD with clipFS (clip, res 224) ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

  loaded: {'missing': 0, 'unexpected': 0}
  scored 64/2493
  scored 704/2493
  scored 1344/2493
  scored 1984/2493
  saved /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/reports/scores/clipFS_DFD.parquet  (2493 frames)  overall AUC=0.813

All 3 detectors scored. Next: CELL E (within-DFD coupling).


## Cell 5 — Build manifest from cropped DFD frames, score, save

Each condition (DFD_c23, DFD_c40) becomes a row in the coupling analysis. We can also split by
manipulation method if DFD provides it, but compression alone gives a competence spread.


In [18]:
# ----------------------------------------------------------------------------
# CELL E — the DFD coupling: does AUC-vs-ECE hold on this 4th dataset?
#
# Two views:
#   (i)  DFD-as-one-point per detector, to overlay on the cross-dataset line
#        (the PRIMARY external-validity test).
#   (ii) per-actor within-DFD coupling, IF actors have both classes and the
#        competence range is non-trivial (SECONDARY; a flat r over a narrow
#        AUC range is expected, not disconfirming — cf. Section 4.2).
#
# Calibration uses the SAME oracle protocol as the paper: 50/50 identity-disjoint
# split, hybrid Platt/isotonic, equal-mass ECE (15 bins).
# ----------------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, glob, os
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from scipy.stats import pearsonr, spearmanr

OUT_SCORES = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/reports/scores"

def ece_equal_mass(p, y, n_bins=15):
    p = np.asarray(p, float); y = np.asarray(y, float)
    o = np.argsort(p); p, y = p[o], y[o]; e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b) == 0: continue
        e += (len(b)/len(p)) * abs(p[b].mean() - y[b].mean())
    return e

def id_split(df, seed=42):
    ids = df["identity_id"].astype(str).to_numpy(); uids = np.unique(ids)
    rng = np.random.RandomState(seed); rng.shuffle(uids); h = len(uids)//2
    return df[np.isin(ids, list(set(uids[:h])))], df[np.isin(ids, list(set(uids[h:])))]

def oracle_ece(df, seed=42):
    """Oracle ECE: fit hybrid calibrator on identity-disjoint half, eval on other half."""
    cal, ev = id_split(df, seed)
    if cal["label"].nunique() < 2 or ev["label"].nunique() < 2:
        # fall back to a random split if identity split collapses a class
        ev_idx = df.sample(frac=0.5, random_state=seed).index
        ev = df.loc[ev_idx]; cal = df.drop(ev_idx)
        if cal["label"].nunique() < 2 or ev["label"].nunique() < 2:
            return np.nan, np.nan
    if len(cal) < 1000:
        m = LogisticRegression(C=1e6); m.fit(cal[["prob_fake"]].values, cal["label"].values)
        cp = m.predict_proba(ev[["prob_fake"]].values)[:, 1]
    else:
        m = IsotonicRegression(out_of_bounds="clip"); m.fit(cal["prob_fake"].values, cal["label"].values)
        cp = m.predict(ev["prob_fake"].values)
    return roc_auc_score(ev["label"], ev["prob_fake"]), ece_equal_mass(cp, ev["label"].values)

print("="*64)
print("DFD COUPLING — external validity on a 4th dataset")
print("="*64)

# ---- VIEW (i): DFD as one point per detector ----
print("\n(i) DFD-as-one-point per detector (overlay on the cross-dataset line):")
print(f"    {'detector':<14}{'AUC':>7}{'oracle ECE':>12}")
dfd_points = []
for f in sorted(glob.glob(f"{OUT_SCORES}/*_DFD.parquet")):
    tag = os.path.basename(f).replace("_DFD.parquet", "")
    d = pd.read_parquet(f)
    if d["label"].nunique() < 2: continue
    auc, ece = oracle_ece(d)
    dfd_points.append({"detector": tag, "AUC": auc, "ECE_cal": ece, "n_frames": len(d)})
    print(f"    {tag:<14}{auc:>7.3f}{ece:>12.3f}")
dfd_df = pd.DataFrame(dfd_points)

# Compare to where the cross-dataset line predicts DFD should sit.
# Load the existing 32-config grid; fit its line; check if DFD points fall near it.
grid_path = f"{OUT_SCORES}/../calibration/coupling_full_grid.csv"
if os.path.exists(grid_path):
    grid = pd.read_csv(grid_path)
    b, a = np.polyfit(grid["AUC"], grid["ECE_cal"], 1)
    print(f"\n    Cross-dataset line (from 32 configs): ECE = {a:.3f} + ({b:.3f})*AUC")
    print(f"    {'detector':<14}{'AUC':>7}{'DFD ECE':>10}{'predicted':>11}{'residual':>10}")
    for _, r in dfd_df.iterrows():
        pred = a + b*r["AUC"]
        print(f"    {r['detector']:<14}{r['AUC']:>7.3f}{r['ECE_cal']:>10.3f}{pred:>11.3f}{r['ECE_cal']-pred:>10.3f}")
    print("    -> small residuals = DFD sits ON the established coupling line (external validity).")

# ---- VIEW (ii): per-actor within-DFD coupling (secondary) ----
print("\n(ii) Per-actor within-DFD coupling (secondary; needs both classes per actor):")
for f in sorted(glob.glob(f"{OUT_SCORES}/*_DFD.parquet")):
    tag = os.path.basename(f).replace("_DFD.parquet", "")
    d = pd.read_parquet(f)
    rows = []
    for actor, g in d.groupby("identity_id"):
        if g["label"].nunique() < 2 or len(g) < 20:   # need both classes + enough frames
            continue
        auc = roc_auc_score(g["label"], g["prob_fake"])
        ece = ece_equal_mass(g["prob_fake"].values, g["label"].values)  # raw ECE per actor (small n)
        rows.append((actor, auc, ece, len(g)))
    if len(rows) >= 5:
        R = pd.DataFrame(rows, columns=["actor", "AUC", "ECE", "n"])
        r, p = pearsonr(R["AUC"], R["ECE"])
        rho, _ = spearmanr(R["AUC"], R["ECE"])
        print(f"    {tag}: n_actors={len(R)}, AUC range [{R.AUC.min():.2f}, {R.AUC.max():.2f}], "
              f"r={r:.3f} (p={p:.3f}), rho={rho:.3f}")
        if R.AUC.max() - R.AUC.min() < 0.15:
            print(f"      NOTE: narrow AUC range -> a flat r is expected, not disconfirming (cf. Sec 4.2)")
    else:
        print(f"    {tag}: too few actors with both classes for a within-DFD correlation")

# save the DFD coupling points for the manuscript
dfd_df.to_csv(f"{OUT_SCORES}/../calibration/dfd_coupling_points.csv", index=False)
print(f"\nsaved dfd_coupling_points.csv")
print("\nINTERPRETATION GUIDE:")
print("  - If the 3 DFD points sit ON the cross-dataset line (small residuals),")
print("    the coupling generalizes to a 4th dataset -> add to Section 4.4.")
print("  - If DFD competence is uniformly high (narrow range), within-DFD r may")
print("    be flat; that is EXPECTED and not evidence against the coupling.")
print("  - If DFD points are clear OUTLIERS off the line, report that honestly;")
print("    do not force-fit. DFD is nice-to-have, not load-bearing.")


DFD COUPLING — external validity on a 4th dataset

(i) DFD-as-one-point per detector (overlay on the cross-dataset line):
    detector          AUC  oracle ECE
    clipFS          0.832       0.062
    effnetb4FS      0.704       0.138
    xceptionFS      0.588       0.127

    Cross-dataset line (from 32 configs): ECE = 0.451 + (-0.446)*AUC
    detector          AUC   DFD ECE  predicted  residual
    clipFS          0.832     0.062      0.080    -0.017
    effnetb4FS      0.704     0.138      0.137     0.001
    xceptionFS      0.588     0.127      0.189    -0.062
    -> small residuals = DFD sits ON the established coupling line (external validity).

(ii) Per-actor within-DFD coupling (secondary; needs both classes per actor):
    clipFS: n_actors=16, AUC range [0.24, 1.00], r=-0.626 (p=0.009), rho=-0.811
    effnetb4FS: n_actors=16, AUC range [0.14, 1.00], r=-0.772 (p=0.000), rho=-0.761
    xceptionFS: n_actors=16, AUC range [0.17, 1.00], r=-0.887 (p=0.000), rho=-0.856

saved dfd_co

In [ ]:
import os, glob
import pandas as pd, numpy as np
CDTS="/content/drive/MyDrive/CDTS_Research"; REPO=f"{CDTS}/deepfake-trust-research"
FRAMES_OUT=f"{REPO}/data/frames/DFD"
import torch, cv2
MEAN=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device);STD=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
def load_img(p): im=cv2.imread(p)[:,:,::-1];im=cv2.resize(im,(256,256));return im.astype(np.float32)/255.0
def to_tensor(im): return (torch.from_numpy(im).permute(2,0,1).unsqueeze(0).to(device)-MEAN)/STD

@torch.no_grad()
def score_frames(paths):
    out=[]
    for i in range(0,len(paths),64):
        batch=paths[i:i+64]
        xs=torch.cat([to_tensor(load_img(p)) for p in batch],0)
        o=model({'image':xs},inference=True)
        pr=o['prob'].cpu().numpy()
        out.extend(pr.tolist() if pr.ndim>0 else [float(pr)])
    return np.array(out)

rows=[]
SCORES_OUT=f"{REPO}/reports/scores/xceptionFS_DFD.parquet"
allscores=[]
for cond in sorted(glob.glob(f"{FRAMES_OUT}/*")):
    condname=os.path.basename(cond)
    fake_p=glob.glob(f"{cond}/fake/*.png"); real_p=glob.glob(f"{cond}/real/*.png")
    if len(fake_p)<20 or len(real_p)<20:
        print(f"{condname}: too few frames (fake={len(fake_p)},real={len(real_p)}), skip"); continue
    pf=score_frames(fake_p); pr=score_frames(real_p)
    p=np.concatenate([pf,pr]); y=np.concatenate([np.ones(len(pf)),np.zeros(len(pr))])
    # identity from filename prefix (actor id) for leakage-safe split
    ids=[os.path.basename(x).split('_')[0] for x in fake_p+real_p]
    dfc=pd.DataFrame({'prob_fake':p,'label':y.astype(int),'identity_id':ids,'condition':condname})
    allscores.append(dfc)
    print(f"{condname}: {len(pf)} fake + {len(pr)} real scored")
scores=pd.concat(allscores,ignore_index=True); scores.to_parquet(SCORES_OUT,index=False)
print(f"\nsaved {len(scores)} DFD scores across {scores['condition'].nunique()} conditions")

## Cell 6 — Per-condition coupling: AUC vs ECE_cal + reference-free signals + commit

In [ ]:
import os, sys, importlib.util
import pandas as pd, numpy as np
from scipy.stats import pearsonr
CDTS="/content/drive/MyDrive/CDTS_Research"; REPO=f"{CDTS}/deepfake-trust-research"
# calibration env
for k in list(sys.modules.keys()):
    if k in ("metrics","calibration") or k.startswith("metrics."): del sys.modules[k]
DFB=f"{REPO}/external/DeepfakeBench"
for p in (f"{DFB}/training",DFB,f"{REPO}/src"):
    if p in sys.path: sys.path.remove(p)
sys.path.insert(0,f"{REPO}/src")
import metrics as metc, calibration as cal

scores=pd.read_parquet(f"{REPO}/reports/scores/xceptionFS_DFD.parquet")
eps=1e-7
rows=[]
# we want MULTIPLE points for the coupling. Use (condition) and also sub-split each condition by
# actor-half to get more competence points if conditions are few.
for cond, d in scores.groupby('condition'):
    if d['label'].nunique()<2: continue
    p=d['prob_fake'].values.astype(float); y=d['label'].values.astype(int)
    g=d['identity_id'].values
    try:
        ci,ti,_=cal.leakage_safe_split(y,groups=g,calib_frac=0.5,seed=42)
        pcal,_=cal.fit_predict("hybrid",p[ci],y[ci],p[ti],switch_threshold_n=1000)
        auc=metc.roc_auc(p[ti],y[ti]); ece=metc.ece(pcal,y[ti],15,'equal_mass')
    except Exception as e:
        print(f"{cond}: calib fail {str(e)[:50]}"); continue
    pc=np.clip(p,eps,1-eps); ent=float(np.mean(-(pc*np.log2(pc)+(1-pc)*np.log2(1-pc))))
    rows.append({'condition':cond,'n':len(d),'AUC':round(auc,4),'ECE_cal':round(ece,4),
                 'entropy':round(ent,4),'dispersion':round(float(np.std(p)),4)})
res=pd.DataFrame(rows)
print("=== DFD per-condition results ===")
print(res.to_string(index=False))

if len(res)>=3:
    r,pp=pearsonr(res['AUC'],res['ECE_cal'])
    print(f"\n=== DFD COUPLING: r(AUC, ECE_cal) = {r:.3f} (p={pp:.3f}, n={len(res)} conditions) ===")
    print(f"    Compare: FF++ r=-0.82, DF40-within-FRmismatch r=-0.94")
    rd,_=pearsonr(res['dispersion'],res['AUC']); print(f"    dispersion vs AUC: r={rd:.3f}")
else:
    print(f"\nOnly {len(res)} conditions - need finer sub-splitting for a coupling correlation.")
    print("(If only 2 compression levels, split each by actor-half or by manipulation method.)")

res.to_csv(f"{REPO}/reports/calibration/coupling_dfd_xceptionFS.csv",index=False)
print("\nsaved coupling_dfd_xceptionFS.csv")

import subprocess
os.chdir(REPO)
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"{CDTS}/{f}"): subprocess.run(f'cp "{CDTS}/{f}" /root/{f}', shell=True)
subprocess.run("git add reports/calibration/coupling_dfd_xceptionFS.csv notebooks/NB15_dfd_coupling.ipynb", shell=True)
print(subprocess.run("git status --short",shell=True,capture_output=True,text=True).stdout)